# BirdCLEF+ 2026 Submission

Loads a trained model checkpoint, runs inference on test soundscapes, and produces `submission.csv`.

In [ ]:
import os, sys
sys.path.insert(0, '/kaggle/input/birdclef-2026-wheels/pkg')

import numpy as np
import pandas as pd
import soundfile as sf
import torch
import torch.nn as nn

from birdclef_2026.data.transforms import build_spectrogram_pipeline
from birdclef_2026.experiments.baseline.model import build_efficientnet_b3_backbone, build_head

SAMPLE_RATE = 32000
CHUNK_SECONDS = 5
CHUNK_SAMPLES = SAMPLE_RATE * CHUNK_SECONDS

COMPETITION_DIR = '/kaggle/input/competitions/birdclef-2026'
CHECKPOINT_PATH = '/kaggle/input/birdclef-2026-checkpoint/checkpoint.pt'
TAXONOMY_PATH = os.path.join(COMPETITION_DIR, 'taxonomy.csv')
TEST_SOUNDSCAPE_DIR = os.path.join(COMPETITION_DIR, 'test_soundscapes')

# Class labels from taxonomy — canonical ordering must match training
taxonomy = pd.read_csv(TAXONOMY_PATH)
class_labels = sorted(taxonomy['primary_label'].astype(str).tolist())
label2idx = {label: i for i, label in enumerate(class_labels)}
n_classes = len(class_labels)
print(f'{n_classes} classes')

In [ ]:
import timm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# --- Transform pipeline (matches training) ---
transform = build_spectrogram_pipeline().to(device)
transform.eval()

# --- Model (must match training config) ---
backbone = timm.create_model('efficientnet_b3', pretrained=False, in_chans=1, num_classes=0)
head = build_head(backbone.num_features, n_classes, hidden=1028)
model = nn.Sequential(backbone, head).to(device)

ckpt = torch.load(CHECKPOINT_PATH, map_location=device)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
print(f'Loaded checkpoint from {CHECKPOINT_PATH}')

In [ ]:
# --- Inference loop ---
test_soundscapes = sorted(
    f for f in os.listdir(TEST_SOUNDSCAPE_DIR) if f.endswith('.ogg')
)
print(f'{len(test_soundscapes)} test soundscapes')

rows = []
for filename in test_soundscapes:
    filepath = os.path.join(TEST_SOUNDSCAPE_DIR, filename)
    sig, rate = sf.read(filepath, dtype='float32')
    if rate != SAMPLE_RATE:
        raise ValueError(f'{filename}: expected {SAMPLE_RATE}Hz, got {rate}Hz')
    soundscape_id = filename.split('.')[0]
    
    for i in range(0, len(sig), CHUNK_SAMPLES):
        chunk = sig[i:i + CHUNK_SAMPLES]
        if len(chunk) < CHUNK_SAMPLES:
            chunk = np.pad(chunk, (0, CHUNK_SAMPLES - len(chunk)))
        
        waveform = torch.from_numpy(chunk).unsqueeze(0).to(device)
        with torch.no_grad():
            image = transform(waveform)
            logits = model(image)
            scores = torch.sigmoid(logits).squeeze(0).cpu().numpy()
        
        end_time = (i // CHUNK_SAMPLES + 1) * CHUNK_SECONDS
        row_id = f'{soundscape_id}_{end_time}'
        rows.append([row_id] + scores.tolist())

submission = pd.DataFrame(rows, columns=['row_id'] + class_labels)
submission.to_csv('submission.csv', index=False)
print(f'submission.csv: {len(submission)} rows x {len(submission.columns)} columns')
submission.head()